# 04 Split LMS Pendikom


In [ ]:
# Jalur REVISED dan pembaca hasil praproses.
from pathlib import Path
import os, sys
from IPython.display import display
try:
    from google.colab import drive
except ImportError:
    pass
else:
    if not Path('/content/drive/MyDrive').is_dir():drive.mount('/content/drive',force_remount=False)
_explicit=os.environ.get('CARE_KM_ROOT') or os.environ.get('CARE_KM_PAPER_ROOT')
_options=([Path(_explicit)] if _explicit else []) + [Path('/content/drive/MyDrive/REVISED'),
    Path(__file__).resolve().parent.parent if '__file__' in globals() else Path.cwd(),Path.cwd(),Path.cwd().parent]
_ROOT_REVISED=next((p.resolve() for p in _options if all((p/d).is_dir() for d in ('DATA','LIB','PYTHON'))),None)
if _ROOT_REVISED is None:raise FileNotFoundError('Letakkan DATA, LIB, PYTHON di MyDrive/REVISED.')
sys.path.insert(0,str(_ROOT_REVISED/'LIB'))
# Jangan memakai modul versi lama yang masih tersimpan di memori Colab.
import importlib
_loaded=sys.modules.get('pipeline_paths')
if _loaded is None or (getattr(_loaded,'PIPELINE_LAYOUT_VERSION',None)!='2026.11.06-experiments-3'
    or Path(getattr(_loaded,'__file__','')).resolve()!=(_ROOT_REVISED/'LIB/pipeline_paths.py').resolve()):
    for _mod in ('pipeline_paths','pipeline_data','pipeline_runner','pipeline_cache','experiment_lib','care_km','generalization_data'):
        sys.modules.pop(_mod,None)
    importlib.invalidate_caches()
from pipeline_paths import configure,stage_dir,lms_data,pendikom_data,artifact_path,record_resample
configure(_ROOT_REVISED)


In [ ]:
from pipeline_data import prepare_splits
from pipeline_paths import record_resample,lms_data
import numpy as np,pandas as pd
folder=prepare_splits(_ROOT_REVISED)
n=len(pd.read_csv(lms_data('categorical')))
settings=[('08_Lintasan_Refinement_CARE_KM',5161600,20),('09_Robustness_LMS_30x80',6161600,30),
          ('13_Stabilitas_Semantik',6161600,30),('16_Sensitivitas_Candidate_Budget',5161600,20),
          ('17_Ablasi_Komponen_CARE_KM',5161600,20)]
for stem,seed,repetitions in settings:
    for rep in range(repetitions):
        indices=np.sort(np.random.default_rng(seed+rep).choice(n,size=int(n*0.8),replace=False))
        record_resample(stem,'LMS',rep,indices)
print('Pembagian holdout dan indeks resampling tersimpan:',folder)
